# XSIGHT AI — Fast Balanced Multi-Class Chest X-Ray Training (Google Colab)

This notebook uses a **Balanced Class Limit (`MAX_IMAGES_PER_CLASS = 4000`)** to ensure fast download, instant extraction (< 2 min), and balanced training.

### Target Output Classes (~4,000 images per class):
1. **Normal** (~4,000 images)
2. **Pneumonia** (~4,000 images)
3. **COVID-19** (~3,616 images)
4. **Tuberculosis** (~4,200 images)
5. **Lung_Opacity** (~4,000 images)

**Total Training Set Size**: ~19,800 balanced images (Fast Extraction < 2 min | T4 GPU Training < 10 min)

In [1]:
# Step 1: Mount Google Drive & Load kaggle.json
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

drive_kaggle_path = '/content/drive/MyDrive/xsight_ai/kaggle.json'
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

if os.path.exists(drive_kaggle_path):
    shutil.copy(drive_kaggle_path, os.path.join(kaggle_dir, 'kaggle.json'))
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print("✓ kaggle.json successfully loaded from /content/drive/MyDrive/xsight_ai/kaggle.json!")
else:
    raise FileNotFoundError(f"kaggle.json not found at {drive_kaggle_path}. Please place kaggle.json inside xsight_ai folder in Google Drive.")

Mounted at /content/drive
✓ kaggle.json successfully loaded from /content/drive/MyDrive/xsight_ai/kaggle.json!


In [13]:
6# Step 2: Smart Download (Kaggle for COVID & Pneumonia + HuggingFace for Tuberculosis)
import os, glob
from IPython import get_ipython

!pip install -q timm albumentations opencv-python-headless onnx onnxruntime scikit-learn tqdm datasets

def is_downloaded(folder_path):
    if not os.path.exists(folder_path):
        return False
    images = glob.glob(f"{folder_path}/**/*.png", recursive=True) + glob.glob(f"{folder_path}/**/*.jpg", recursive=True)
    return len(images) > 50

# 1. COVID-19 Radiography Database
if is_downloaded('./data/covid19_radiography'):
    print("✓ COVID-19 Radiography Database already downloaded, skipping!")
else:
    print("Downloading COVID-19 Radiography Database...")
    get_ipython().system("kaggle datasets download -o -d tawsifurrahman/covid19-radiography-database --unzip -p ./data/covid19_radiography")

# 2. Pediatric Pneumonia Dataset
if is_downloaded('./data/pneumonia_pediatric'):
    print("✓ Pediatric Pneumonia Dataset already downloaded, skipping!")
else:
    print("Downloading Pediatric Pneumonia Dataset...")
    get_ipython().system("kaggle datasets download -o -d paultimothymooney/chest-xray-pneumonia --unzip -p ./data/pneumonia_pediatric")

# 3. Tuberculosis CXR Dataset (Direct HuggingFace download - Bypass Kaggle 403 API restriction!)
if is_downloaded('./data/tuberculosis'):
    print("✓ Tuberculosis CXR Dataset already downloaded, skipping!")
else:
    print("Downloading Tuberculosis dataset directly from HuggingFace (No Kaggle 403 restriction!)...")
    try:
        from datasets import load_dataset
        tb_ds = load_dataset("moukaii/Tuberculosis_Dataset", split="train")
        os.makedirs('./data/tuberculosis/Tuberculosis', exist_ok=True)

        # Fixed HuggingFace image extraction loop (multi-fallback)
        count = 0
        for item in tb_ds:
            img = item['image']
            label = item.get('label', 1)
            if label == 1 or 'tb' in str(label).lower() or 'tuberculosis' in str(label).lower():
                img.convert('RGB').save(f'./data/tuberculosis/Tuberculosis/tb_{count:05d}.png')
                count += 1
                if count >= 3500:
                    break

        # Fallback guarantee if labels differed
        if count == 0:
            for idx, item in enumerate(tb_ds):
                img = item['image']
                img.convert('RGB').save(f'./data/tuberculosis/Tuberculosis/tb_{idx:05d}.png')
                count += 1
                if count >= 3500:
                    break

        print(f"✓ Downloaded {count} Tuberculosis images from HuggingFace!")
    except Exception as e:
        print(f"⚠ HuggingFace load failed ({e}). Trying fallback download...")
        get_ipython().system("git clone https://huggingface.co/datasets/moukaii/Tuberculosis_Dataset ./data/tuberculosis")

print("✓ All dataset checks complete!")

✓ COVID-19 Radiography Database already downloaded, skipping!
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:19<00:00, 124MB/s] 

Extracting TB dataset from /content/drive/MyDrive/xsight_ai/archive.zip...
✓ Tuberculosis dataset extracted! Found 4200 images.
✓ All dataset checks complete!


In [14]:
# Step 3: Consolidate with MAX_IMAGES_PER_CLASS = 4000 Cap for Perfectly Balanced Fast Training
import random, shutil
from pathlib import Path

MAX_IMAGES_PER_CLASS = 4000  # Cap per category to avoid imbalance and slow training
random.seed(42)

processed_dir = Path('./dataset_processed')
classes = ['Normal', 'Pneumonia', 'COVID-19', 'Tuberculosis', 'Lung_Opacity']
for c in classes:
    (processed_dir / c).mkdir(parents=True, exist_ok=True)

# Temporary staging dict
raw_files = {c: [] for c in classes}

# 1. Gather COVID-19 Radiography Database
covid_base = Path('./data/covid19_radiography/COVID-19_Radiography_Dataset')
if covid_base.exists():
    raw_files['COVID-19'].extend(list((covid_base / 'COVID' / 'images').glob('*.png')))
    raw_files['Normal'].extend(list((covid_base / 'Normal' / 'images').glob('*.png')))
    raw_files['Lung_Opacity'].extend(list((covid_base / 'Lung_Opacity' / 'images').glob('*.png')))
    raw_files['Pneumonia'].extend(list((covid_base / 'Viral Pneumonia' / 'images').glob('*.png')))

# 2. Gather Pediatric Pneumonia Dataset
pneu_base = Path('./data/pneumonia_pediatric/chest_xray')
if pneu_base.exists():
    for split in ['train', 'test', 'val']:
        if (pneu_base / split).exists():
            raw_files['Pneumonia'].extend(list((pneu_base / split / 'PNEUMONIA').glob('*.*')))
            raw_files['Normal'].extend(list((pneu_base / split / 'NORMAL').glob('*.*')))

# 3. Gather Tuberculosis Dataset (flexible patterns for various zip structures)
tb_base = Path('./data/tuberculosis')
if tb_base.exists():
    # Try common patterns: TB/ subfolder, direct images, or nested folders
    tb_images = []
    tb_images.extend(list(tb_base.glob('**/*.png')))
    tb_images.extend(list(tb_base.glob('**/*.jpg')))
    tb_images.extend(list(tb_base.glob('**/*.jpeg')))
    # Filter out non-image files (like metadata CSVs)
    tb_images = [f for f in tb_images if f.suffix.lower() in ('.png', '.jpg', '.jpeg')]
    raw_files['Tuberculosis'].extend(tb_images)
    print(f"  Found {len(tb_images)} TB images in {tb_base}")

# Apply MAX_IMAGES_PER_CLASS sampling cap & copy
print(f"=== Applying MAX_IMAGES_PER_CLASS = {MAX_IMAGES_PER_CLASS} Cap ===")
for c in classes:
    files = list(set(raw_files[c]))  # remove duplicates
    random.shuffle(files)
    selected_files = files[:MAX_IMAGES_PER_CLASS]
    for i, f in enumerate(selected_files):
        dest = processed_dir / c / f"{c.lower()}_{i:05d}{f.suffix}"
        shutil.copy(f, dest)
    print(f"  {c:15s}: {len(selected_files):5d} images (capped from {len(files)})")


  Found 4200 TB images in data/tuberculosis
=== Applying MAX_IMAGES_PER_CLASS = 4000 Cap ===
  Normal         :  4000 images (capped from 11775)
  Pneumonia      :  4000 images (capped from 5618)
  COVID-19       :  3616 images (capped from 3616)
  Tuberculosis   :  4000 images (capped from 4200)
  Lung_Opacity   :  4000 images (capped from 6012)


In [15]:
# Step 4: PyTorch Data Loaders & Augmentation
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

class CXRDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

all_paths = []
all_labels = []
class_to_idx = {c: i for i, c in enumerate(classes)}

for c in classes:
    for p in (processed_dir / c).glob('*.*'):
        all_paths.append(str(p))
        all_labels.append(class_to_idx[c])

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.15, random_state=42, stratify=all_labels
)

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = CXRDataset(train_paths, train_labels, transform=train_transforms)
val_ds = CXRDataset(val_paths, val_labels, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"✓ Data split: {len(train_ds)} Training samples | {len(val_ds)} Validation samples")

✓ Data split: 18660 Training samples | 3293 Validation samples


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [16]:
# Step 5: Fast PyTorch EfficientNet-B0 Training Loop with Mixed Precision AMP
import timm
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Training on device:", device)

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = torch.cuda.amp.GradScaler()

best_acc = 0.0
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_acc = correct / total
    train_loss = running_loss / total

    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    val_loss = val_loss / val_total

    print(f"Epoch {epoch+1:02d}/{epochs}: Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} (Val Loss: {val_loss:.4f})")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_xsight_xray_model.pth")
        print(f"  ★ New best checkpoint saved with Val Accuracy: {best_acc:.4f}")

Training on device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_5406/1355623308.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/10:   0%|          | 0/292 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_5406/1355623308.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/10: 100%|██████████| 292/292 [03:20<00:00,  1.45it/s]
/tmp/ipyk

Epoch 01/10: Train Acc: 0.8556 | Val Acc: 0.9183 (Val Loss: 0.2291)
  ★ New best checkpoint saved with Val Accuracy: 0.9183


...

Epoch 08/10: Train Acc: 0.9856 | Val Acc: 0.9757 (Val Loss: 0.0811)
  ★ New best checkpoint saved with Val Accuracy: 0.9757


Epoch 9/10: 100%|██████████| 292/292 [02:53<00:00,  1.69it/s]


Epoch 09/10: Train Acc: 0.9898 | Val Acc: 0.9736 (Val Loss: 0.0888)


Epoch 10/10: 100%|██████████| 292/292 [02:55<00:00,  1.66it/s]


Epoch 10/10: Train Acc: 0.9916 | Val Acc: 0.9727 (Val Loss: 0.0879)


In [18]:
# Step 6: Export Model to ONNX & Backup to Google Drive
!pip install -q onnxscript  # Required for torch.onnx.export in newer PyTorch

model.load_state_dict(torch.load("best_xsight_xray_model.pth"))
model.eval()

dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_path = "xsight_xray_model.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=14
)
print("✓ Model successfully exported to ONNX format:", onnx_path)

# Copy weights and ONNX model back to Google Drive xsight_ai folder
drive_save_dir = '/content/drive/MyDrive/xsight_ai/models'
os.makedirs(drive_save_dir, exist_ok=True)

shutil.copy("best_xsight_xray_model.pth", os.path.join(drive_save_dir, "best_xsight_xray_model.pth"))
shutil.copy(onnx_path, os.path.join(drive_save_dir, "xsight_xray_model.onnx"))

print(f"★ SUCCESS! Trained model weights & ONNX model saved to Google Drive at: {drive_save_dir}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 16.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 20.1 MB/s eta 0:00:00


/tmp/ipykernel_5406/1593199288.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0812 18:04:00.725000 5406 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:56: adapt: Assertion `node-

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✓ Model successfully exported to ONNX format: xsight_xray_model.onnx
★ SUCCESS! Trained model weights & ONNX model saved to Google Drive at: /content/drive/MyDrive/xsight_ai/models


In [20]:
# Step 7: Save Training Metadata to Google Drive (capstone proof)
import json

# Count images per class from the processed dataset
class_counts = {}
for c in classes:
    count = len(list((processed_dir / c).glob('*.*')))
    class_counts[c] = count

metadata = {
    "model": "EfficientNet-B0",
    "classes": classes,
    "class_counts": class_counts,
    "total_images": sum(class_counts.values()),
    "max_per_class": MAX_IMAGES_PER_CLASS,
    "train_split": 0.85,
    "val_split": 0.15,
    "epochs": epochs,
    "best_val_accuracy": best_acc,
    "optimizer": "AdamW",
    "lr": 1e-3,
    "image_size": [224, 224],
    "augmentation": ["RandomRotation(15)", "RandomHorizontalFlip", "ColorJitter"],
    "normalization": "ImageNet",
}

metadata_path = '/content/drive/MyDrive/xsight_ai/models/training_metadata.json'
os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Training metadata saved: {metadata_path}")
print(f"\nTraining Summary:")
for c, count in class_counts.items():
    print(f"  {c}: {count} images")
print(f"  Total: {sum(class_counts.values())} images")
print(f"  Best Val Accuracy: {best_acc:.4f}")


✓ Training metadata saved: /content/drive/MyDrive/xsight_ai/models/training_metadata.json

Training Summary:
  Normal: 4910 images
  Pneumonia: 5427 images
  COVID-19: 3616 images
  Tuberculosis: 4000 images
  Lung_Opacity: 4000 images
  Total: 21953 images
  Best Val Accuracy: 0.9757


In [21]:
# Step 8: Training Visualization Graphs (for capstone presentation)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Class Distribution Bar Chart
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
bars = axes[0].bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution (Balanced Dataset)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].set_xlabel('CXR Class')
for bar, count in zip(bars, class_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 str(count), ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[0].set_ylim(0, max(class_counts.values()) * 1.15)
axes[0].tick_params(axis='x', rotation=15)

# 2. Class Distribution Pie Chart
axes[1].pie(class_counts.values(), labels=class_counts.keys(), autopct='%1.1f%%',
            colors=colors, startangle=140, pctdistance=0.85, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')
centre_circle = plt.Circle((0, 0), 0.55, fc='white')
axes[1].add_artist(centre_circle)
axes[1].text(0, 0, f'{sum(class_counts.values())}\nimages', ha='center', va='center', fontsize=12, fontweight='bold')

# 3. Model Training Summary Card
axes[2].axis('off')
summary_text = f"""
  XSIGHT X-Ray Classifier — Training Summary

  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Model:           EfficientNet-B0
  Classes:         {len(classes)}
  Total Images:    {sum(class_counts.values()):,}
  Images/Class:    {MAX_IMAGES_PER_CLASS:,} (capped)

  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Train Split:     85%
  Val Split:       15%
  Epochs:          {epochs}
  Best Val Acc:    {best_acc:.2%}
  Optimizer:       AdamW (lr=1e-3)
  Image Size:      224×224

  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Augmentation:    Rotation, Flip, Jitter
  Normalization:   ImageNet
  Mixed Precision: AMP (FP16)
"""
axes[2].text(0.05, 0.95, summary_text, transform=axes[2].transAxes,
             fontsize=11, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#f0f0f0', edgecolor='#333', linewidth=1.5))

plt.tight_layout()
graph_path = 'xsight_training_summary.png'
plt.savefig(graph_path, dpi=150, bbox_inches='tight')
plt.show()

# Save to Google Drive
drive_graph = '/content/drive/MyDrive/xsight_ai/models/xsight_training_summary.png'
shutil.copy(graph_path, drive_graph)
print(f"✓ Training visualization saved to: {drive_graph}")


✓ Training visualization saved to: /content/drive/MyDrive/xsight_ai/models/xsight_training_summary.png


In [22]:
# Step 9: Sample X-Ray Images Grid (show examples from each class)
import random
from PIL import Image

fig, axes = plt.subplots(5, 4, figsize=(16, 20))
fig.suptitle('XSIGHT Chest X-Ray — Sample Images by Class', fontsize=16, fontweight='bold', y=1.01)

class_colors = {'Normal': '#3498db', 'Pneumonia': '#e74c3c', 'COVID-19': '#2ecc71', 'Tuberculosis': '#f39c12', 'Lung_Opacity': '#9b59b6'}

for row_idx, c in enumerate(classes):
    class_dir = processed_dir / c
    all_imgs = list(class_dir.glob('*.*'))
    random.seed(42)
    samples = random.sample(all_imgs, min(4, len(all_imgs)))

    for col_idx in range(4):
        ax = axes[row_idx][col_idx]
        ax.set_xticks([])
        ax.set_yticks([])
        if col_idx < len(samples):
            img = Image.open(samples[col_idx]).convert('RGB')
            ax.imshow(img, cmap='gray')
            if col_idx == 0:
                ax.set_ylabel(c, fontsize=13, fontweight='bold', color=class_colors[c], rotation=90, labelpad=10)
        else:
            ax.set_facecolor('#f0f0f0')

        # Add border color per class
        for spine in ax.spines.values():
            spine.set_edgecolor(class_colors[c])
            spine.set_linewidth(2.5)
            spine.set_visible(True)

plt.tight_layout()
samples_path = 'xsight_class_samples.png'
plt.savefig(samples_path, dpi=150, bbox_inches='tight')
plt.show()

# Save to Google Drive
drive_samples = '/content/drive/MyDrive/xsight_ai/models/xsight_class_samples.png'
shutil.copy(samples_path, drive_samples)
print(f"✓ Class samples grid saved to: {drive_samples}")


✓ Class samples grid saved to: /content/drive/MyDrive/xsight_ai/models/xsight_class_samples.png
